In [1]:
import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord
import spiceypy as spice

from light_deflection import delta_kpn, calc_CE, calc_der_E, calc_der_star, calc_CPE, calc_CO, calc_CT
from ephemeris import ephem_kernel, on_sky_velocity

In [2]:
# Occultation parameters
occ_time = Time('2021-04-02 10:24:00', scale='utc')
occ_vel = 16.54*u.km/u.s
io_radius = 1821*u.km

In [3]:
# Geocentric star position at occultation epoch
# propagated with proper motion, parallax and radial velocity using Butkevich & Lindegren (2014)
pos_star = SkyCoord('21h43m04.38939019s -14d23m58.5335448s', distance=142.9253799674684*u.pc).cartesian

In [4]:
# GM of Jupiter and Sun form JUP365: https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/satellites/jup365.cmt
jup_GM = 1.266865319003704E+08 * (u.km**3) / (u.s**2)
sun_GM = 1.327132332633514E+11 * (u.km**3) / (u.s**2)
# Jupiter and Io SPKID
jup_spkid = '599'
sun_spkid = '10'
io_spkid = '501'

In [5]:
# JUP365 in local server
kernel = ['/srv/jupyterhub/shared/data/kernels/jup365.bsp']
spice.furnsh(kernel)

In [6]:
# Jupiter position at occultation epoch
pos_jup, vel_jup = ephem_kernel(time=occ_time, target=jup_spkid, observer='500')
pos_jup = pos_jup.cartesian

In [7]:
# Sun position at occultation epoch
pos_sun, vel_sun = ephem_kernel(time=occ_time, target=sun_spkid, observer='500')
pos_sun = pos_sun.cartesian

In [8]:
# Io position at occultation epoch
pos_io, vel_io = ephem_kernel(time=occ_time, target=io_spkid, observer='500')
e_f = SkyCoord(-np.sin(pos_io.spherical.lon), np.cos(pos_io.spherical.lon), 0, representation_type='cartesian').cartesian
e_g = SkyCoord(-np.cos(pos_io.spherical.lon)*np.sin(pos_io.spherical.lat), -np.sin(pos_io.spherical.lon)*np.sin(pos_io.spherical.lat), np.cos(pos_io.spherical.lat), representation_type='cartesian').cartesian
pos_io = pos_io.cartesian

In [9]:
spice.kclear()

In [10]:
# Distances
DA_jup = pos_jup.norm()  # massive
DA_sun = pos_sun.norm()  # massive
DE_io = pos_io.norm()   # target
print('Jup', DA_jup.to(u.au))
print('Sun', DA_sun.to(u.au))
print('Io', DE_io.to(u.au))

Jup 5.6591845286255476 AU
Sun 0.9996008691281666 AU
Io 5.656483416298644 AU


In [11]:
# Compute light deflection for the star due to Jupiter
vec_rso = - pos_star
vec_rao = - pos_jup
vec_ras = pos_star - pos_jup
delta_k_star_jup = delta_kpn(gm=jup_GM, vec_rso=vec_rso, vec_rao=vec_rao, vec_ras=vec_ras)
print(delta_k_star_jup.norm().to(u.mas, equivalencies=u.dimensionless_angles()))

9.607933245067807 mas


In [12]:
# Compute light deflection for the star due to the Sun
vec_rso = - pos_star
vec_rao = - pos_sun
vec_ras = pos_star - pos_sun
delta_k_star_sun = delta_kpn(gm=sun_GM, vec_rso=vec_rso, vec_rao=vec_rao, vec_ras=vec_ras)
print(delta_k_star_sun.norm().to(u.mas, equivalencies=u.dimensionless_angles()))

8.855870063374532 mas


In [13]:
# Compute light deflection for the ephemeris position due to Jupiter
vec_rso = - pos_io
vec_rao = - pos_jup
vec_ras = pos_io - pos_jup
delta_k_E_jup = delta_kpn(gm=jup_GM, vec_rso=vec_rso, vec_rao=vec_rao, vec_ras=vec_ras)
print(delta_k_E_jup.norm().to(u.mas, equivalencies=u.dimensionless_angles()))

0.00010075757745497693 mas


In [14]:
# Compute light deflection for the ephemeris position due the Sun
vec_rso = - pos_io
vec_rao = - pos_sun
vec_ras = pos_io - pos_sun
delta_k_E_sun = delta_kpn(gm=sun_GM, vec_rso=vec_rso, vec_rao=vec_rao, vec_ras=vec_ras)
print(delta_k_E_sun.norm().to(u.mas, equivalencies=u.dimensionless_angles()))

7.345141532854214 mas


In [15]:
phi_E_jup = SkyCoord(pos_jup).separation(SkyCoord(pos_io))
phi_star_jup = SkyCoord(pos_jup).separation(SkyCoord(pos_star))
print(phi_E_jup.arcsec, phi_star_jup.arcsec)

29.533918677061017 29.4910324916616


In [16]:
phi_E_sun = SkyCoord(pos_sun).separation(SkyCoord(pos_io))
phi_star_sun = SkyCoord(pos_sun).separation(SkyCoord(pos_star))
print(phi_E_sun.deg, phi_star_sun.deg)

49.40355844147594 49.40354150206593


In [19]:
delta_theta_jup = delta_k_E_jup - delta_k_star_jup
delta_theta_jup_f = delta_theta_jup.dot(e_f)
delta_theta_jup_g = delta_theta_jup.dot(e_g)
print(delta_theta_jup_f.to(u.mas, equivalencies=u.dimensionless_angles()),
      delta_theta_jup_g.to(u.mas, equivalencies=u.dimensionless_angles()))

-8.870560720949676 mas -3.6910158235837356 mas


In [20]:
delta_theta_sun = delta_k_E_sun - delta_k_star_sun
delta_theta_sun_f = delta_theta_sun.dot(e_f)
delta_theta_sun_g = delta_theta_sun.dot(e_g)
print(delta_theta_sun_f.to(u.mas, equivalencies=u.dimensionless_angles()),
      delta_theta_sun_g.to(u.mas, equivalencies=u.dimensionless_angles()))

-1.421754027422862 mas -0.5107995483945954 mas


In [32]:
delta_theta_tot = (delta_k_E_jup + delta_k_E_sun) - (delta_k_star_jup + delta_k_star_sun)
delta_theta_tot_f = delta_theta_tot.dot(e_f)
delta_theta_tot_g = delta_theta_tot.dot(e_g)
print(delta_theta_tot_f.to(u.mas, equivalencies=u.dimensionless_angles()),
      delta_theta_tot_g.to(u.mas, equivalencies=u.dimensionless_angles()))

-10.292314748372538 mas -4.201815371978331 mas


In [21]:
dot_alpha_jup, dot_delta_jup = on_sky_velocity(pos_jup, vel_jup)
dot_alpha_sun, dot_delta_sun = on_sky_velocity(pos_sun, vel_sun)
dot_alpha_io, dot_delta_io = on_sky_velocity(pos_io, vel_io)
print('Io', dot_alpha_io.to(u.arcsec/u.h), dot_delta_io.to(u.arcsec/u.h))
print('Jup', dot_alpha_jup.to(u.arcsec/u.h), dot_delta_jup.to(u.arcsec/u.h))
print('Sun', dot_alpha_sun.to(u.arcsec/u.h), dot_delta_sun.to(u.arcsec/u.h))

Io 13.988729579552382 arcsec / h 3.8399167556613465 arcsec / h
Jup 27.562337338463724 arcsec / h 9.148990859313523 arcsec / h
Sun 136.22028224218658 arcsec / h 57.62019377940502 arcsec / h


In [22]:
dot_phi_star_jup = np.sqrt(dot_alpha_jup**2 + dot_delta_jup**2)
dot_phi_star_sun = np.sqrt(dot_alpha_sun**2 + dot_delta_sun**2)
print('Jup', dot_phi_star_jup.to(u.arcsec/u.h))
print('Sun', dot_phi_star_sun.to(u.arcsec/u.h))

Jup 29.041116943104544 arcsec / h
Sun 147.9055510294227 arcsec / h


In [23]:
dot_alpha_io_jup = dot_alpha_io - dot_alpha_jup
dot_delta_io_jup = dot_delta_io - dot_delta_jup
dot_phi_io_jup = np.sqrt(dot_alpha_io_jup**2 + dot_delta_io_jup**2)
print(dot_phi_io_jup.to(u.arcsec/u.h))

14.574947527550425 arcsec / h


In [24]:
dot_alpha_io_sun = dot_alpha_io - dot_alpha_sun
dot_delta_io_sun = dot_delta_io - dot_delta_sun
dot_phi_io_sun = np.sqrt(dot_alpha_io_sun**2 + dot_delta_io_sun**2)
print(dot_phi_io_sun.to(u.arcsec/u.h))

133.53977183996133 arcsec / h


In [25]:
der_E_jup = calc_der_E(gm=jup_GM, DA=DA_jup, DE=DE_io, phi_E=phi_E_jup)
der_star_jup = calc_der_star(gm=jup_GM, DA=DA_jup, phi_star=phi_star_jup)

In [26]:
der_E_sun = calc_der_E(gm=sun_GM, DA=DA_sun, DE=DE_io, phi_E=phi_E_sun)
der_star_sun = calc_der_star(gm=sun_GM, DA=DA_sun, phi_star=phi_star_sun)

In [27]:
# Compute Light deflection Correction Required for the tangente plane origin at t=t_occ
CE_jup = calc_CE(DE_io, delta_k_E_jup, delta_k_star_jup)
CE_jup.to(u.km)

<Quantity 39.41597012 km>

In [28]:
# Compute Light deflection Correction Required for the tangente plane origin at t=t_occ
CE_sun = calc_CE(DE_io, delta_k_E_sun, delta_k_star_sun)
CE_sun.to(u.km)

<Quantity 6.19773821 km>

In [29]:
# Compute Light deflection Correction Required for the tangente plane origin at t=t_occ
CE_tot = calc_CE(DE_io, delta_k_E_jup+delta_k_E_sun, delta_k_star_jup+delta_k_star_sun)
CE_tot.to(u.km)

<Quantity 45.60717625 km>

In [25]:
CPE_jup = calc_CPE(der_E_jup, RP=io_radius)
CPE_jup.to(u.m)

<Quantity 0.0059505 m>

In [26]:
CPE_sun = calc_CPE(der_E_sun, RP=io_radius)
CPE_sun.to(u.m)

<Quantity 0.08443636 m>

In [27]:
CO_jup = calc_CO(der_E=der_E_jup, der_star=der_star_jup, DA=DA_jup, DE=DE_io, RO=io_radius)
CO_jup.to(u.m)

<Quantity 592.98360792 m>

In [28]:
CO_sun = calc_CO(der_E=der_E_sun, der_star=der_star_sun, DA=DA_sun, DE=DE_io, RO=io_radius)
CO_sun.to(u.m)

<Quantity 0.97602825 m>

In [29]:
CT_jup = calc_CT(der_E=der_E_jup, der_star=der_star_jup, dot_phi_E_A=dot_phi_io_jup, dot_phi_star_A=dot_phi_star_jup,
                 DE=DE_io, RT=io_radius, v_occ=occ_vel)
CT_jup.to(u.m, equivalencies=u.dimensionless_angles())

<Quantity 1187.06449626 m>

In [30]:
CT_sun = calc_CT(der_E=der_E_sun, der_star=der_star_sun, dot_phi_E_A=dot_phi_io_sun, dot_phi_star_A=dot_phi_star_sun,
                 DE=DE_io, RT=io_radius, v_occ=occ_vel)
CT_sun.to(u.m, equivalencies=u.dimensionless_angles())

<Quantity 1.82614607 m>

In [33]:
delta = np.array([1.1, 0.7])*u.mas

In [35]:
DE_io*(delta.to(u.rad).value)

<Quantity [4.51273137, 2.87173815] km>

In [36]:
DE_io*np.linalg.norm(delta).to(u.rad).value

<Quantity 5.34898349 km>